In [0]:
enc_df = spark.table("medical_project.silver.encounters")
proc_df = spark.table("medical_project.silver.procedures")

In [0]:
from pyspark.sql.functions import sum, count

proc_agg = proc_df.groupBy("encounter_id") \
    .agg(
        sum("base_cost").alias("total_cost"),
        count("*").alias("procedure_count")
    )

display(proc_agg)

In [0]:
fact_df = enc_df.join(
    proc_agg,
    enc_df.id == proc_agg.encounter_id,
    "left"
)

display(fact_df)


In [0]:
from pyspark.sql.functions import col, when

fact_df = fact_df.select(
    col("id").alias("encounter_id"),
    col("patient").alias("patient_id"),
    col("payer").alias("payer_id"),
    col("encounter_class"),
    col("start"),
    col("stop"),
    col("encounter_duration_hours"),
    col("total_cost"),
    col("procedure_count")
)

display(fact_df)

In [0]:
display(fact_df.summary())

In [0]:
# Over 24 Hours Flag

fact_df = fact_df.withColumn(
    "is_over_24_hours",
    when(col("encounter_duration_hours") > 24, 1).otherwise(0)
)

# Payer Coverage Flag

fact_df = fact_df.withColumn(
    "has_payer_coverage",
    when(col("payer_id").isNull(), 0).otherwise(1)
)

display(fact_df)


In [0]:
# Handle Null Cost
fact_df = fact_df.fillna({
    "total_cost": 0,
    "procedure_count": 0
})

In [0]:
fact_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_project.gold.fact_encounters")